In [ ]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ROOT = os.path.dirname(os.getcwd())
TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_clean")
TARGET_DLQ_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_dlq")

builder = (
    SparkSession.builder.appName("Query_Silver")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
# Load your local enriched Silver Delta table
silver_df = spark.read.format("delta").load(TARGET_PATH)

print(f"Total Clean Multi-Region Threat Records: {silver_df.count()}")

# Sort by latest processed to see fresh live traffic at the top
latest_threats = silver_df.orderBy(F.col("timestamp").desc())

# Render the exact fields engineered by your regex and Broadcast Join
latest_threats.select(
    "honeypot_host",
    "honeypot_region",
    "attacker_ip",
    "targeted_username",
    "attacker_country",
    "attacker_country_code",
    "timestamp"
).show(15, truncate=False)